# Modality-budget experiment: full pipeline (Kaggle, BraTS 2021 tar)
Notebook settings: **Accelerator = GPU T4 x2**, Internet **On**, dataset `dschettler8845/brats-2021-task1` added under *Add Input*.

Run the cells in order. **Every command is run through the Python helper `run()` with explicit arguments**, so there are no `$VARIABLE` shell expansions to go wrong.

## 1. Install

In [1]:
!pip install -q "monai>=1.3" nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.3 MB/s eta 0:00:0000:0100:01


## 2. Write the experiment script to disk (complete implementation)

In [2]:
%%writefile modality_budget_experiment.py
#!/usr/bin/env python
"""
Modality-budget experiment for multimodal brain tumor segmentation (BraTS).

Implements the reporting protocol of Section VII of the paper:
  * subset sweep over all 15 non-empty subsets of {T1, T1ce, T2, FLAIR}
  * Shapley marginal utility per sequence and tumor region (ET, TC, WT)
  * budget curves (best / mean / worst subset at each budget size k)
  * cost-weighted minimal sufficient sets at tolerance eps
  * bootstrap-over-cases 95% confidence intervals

Sub-commands
  train     train one SegResNet (mode=dropout: modality-dropout training,
            mode=full: trained on complete input only; use both for a baseline)
  evaluate  run the 15-subset sweep on the held-out test split -> results CSV
  analyze   compute Shapley / budget curves / minimal sets / figures from CSV(s)
  selftest  run `analyze` on synthetic data (no GPU, no MONAI needed)

Designed for Kaggle (2 x T4). Torch/MONAI are imported lazily, so `analyze`
and `selftest` only need numpy, pandas and matplotlib.
"""
import argparse
import itertools
import json
import math
import os
import random
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

MODS = ["T1", "T1ce", "T2", "FLAIR"]          # bit i of a mask <-> MODS[i]
REGIONS = ["ET", "TC", "WT"]
ALL_MASKS = list(range(1, 16))                # 15 non-empty subsets
POPCOUNT = np.array([bin(m).count("1") for m in range(16)])

# Placeholder acquisition costs (relative units). T1ce is weighted higher because it
# needs contrast administration. These are NOT clinical estimates: set them with
# radiology input and report them explicitly (see README).
DEFAULT_COSTS = {"T1": 1.0, "T1ce": 3.0, "T2": 1.0, "FLAIR": 1.0}

if os.path.exists("/kaggle"):
    DEFAULT_ROOT, DEFAULT_OUT = "/kaggle/input", "/kaggle/working/mb_exp"
elif os.path.exists("/content"):                       # Google Colab
    DEFAULT_ROOT, DEFAULT_OUT = "/content", "/content/mb_exp"
else:
    DEFAULT_ROOT, DEFAULT_OUT = ".", "./mb_exp"


def subset_name(mask):
    return "+".join(MODS[i] for i in range(4) if (mask >> i) & 1)


# --------------------------------------------------------------------------- #
# Data discovery and splitting
# --------------------------------------------------------------------------- #
_PATS = {
    "T1": r"[_-]t1n?\.nii(\.gz)?$",
    "T1ce": r"[_-](t1ce|t1gd|t1c)\.nii(\.gz)?$",
    "T2": r"[_-]t2w?\.nii(\.gz)?$",
    "FLAIR": r"[_-](flair|t2f)\.nii(\.gz)?$",
}


def find_cases(root):
    """Find case folders containing the 4 NIfTI sequences + a segmentation.
    Works for BraTS 2018-2021 and 2023 file naming."""
    cases = []
    for dirpath, _, files in os.walk(root):
        low = {f.lower(): f for f in files}
        found = {}
        for mod, pat in _PATS.items():
            hit = [orig for l, orig in low.items() if re.search(pat, l)]
            if hit:
                found[mod] = os.path.join(dirpath, hit[0])
        seg = [orig for l, orig in low.items()
               if "seg" in l and re.search(r"\.nii(\.gz)?$", l)]
        if len(found) == 4 and seg:
            cases.append({"id": os.path.basename(dirpath),
                          "image": [found[m] for m in MODS],
                          "label": os.path.join(dirpath, seg[0])})
    cases.sort(key=lambda c: c["id"])
    return cases


def diagnose(root):
    """Explain why no cases were found: counts file types and shows example paths."""
    ext = {}
    examples = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            e = ".nii.gz" if f.lower().endswith(".nii.gz") else os.path.splitext(f)[1].lower()
            ext[e] = ext.get(e, 0) + 1
            if len(examples) < 6 and e in (".nii", ".nii.gz", ".h5"):
                examples.append(os.path.join(dirpath, f))
    print(f"\n--- data diagnosis for {root} ---")
    print("file types found:", dict(sorted(ext.items(), key=lambda kv: -kv[1])[:8]) or "none")
    print("example files:", *examples, sep="\n  ")
    if ext.get(".nii", 0) + ext.get(".nii.gz", 0) == 0:
        if ext.get(".h5", 0):
            print("-> Only .h5 slices found. This script needs the NIfTI version of BraTS "
                  "(.nii/.nii.gz per sequence).")
        else:
            print("-> No NIfTI files at all. Is the dataset added under 'Add Input'? "
                  "Check `!ls /kaggle/input`.")
    else:
        print("-> NIfTI files exist but no folder has all 4 sequences + a seg file named "
              "like *_t1, *_t1ce/_t1gd, *_t2, *_flair, *_seg. Send me these example names.")


def require_cases(root, max_cases=0, seed=42):
    cases = find_cases(root)
    if not cases:
        diagnose(root)
        sys.exit("\nNo BraTS cases found under " + root + " (see diagnosis above).")
    return subsample_cases(cases, max_cases, seed)


def get_split(cases, out_dir, seed):
    """Fixed 70/10/20 train/val/test split by case id, shared by all models."""
    path = Path(out_dir) / "split.json"
    if path.exists():
        sp = json.loads(path.read_text())
    else:
        ids = [c["id"] for c in cases]
        random.Random(seed).shuffle(ids)
        n_test, n_val = round(0.2 * len(ids)), round(0.1 * len(ids))
        sp = {"test": ids[:n_test], "val": ids[n_test:n_test + n_val],
              "train": ids[n_test + n_val:]}
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(sp, indent=1))
    by_id = {c["id"]: c for c in cases}
    return {k: [by_id[i] for i in v if i in by_id] for k, v in sp.items()}


# --------------------------------------------------------------------------- #
# Transforms (plain callables, so MONAI is only needed at run time)
# --------------------------------------------------------------------------- #
try:                                     # MONAI is optional for `analyze` / `selftest`
    from monai.transforms import Transform as _TfBase
except Exception:                        # pragma: no cover
    _TfBase = object


class LoadBraTS(_TfBase):
    def __call__(self, data):
        import nibabel as nib
        d = dict(data)
        images = []
        spacing = None
        spacing_unit = "voxel"
        for p in d["image"]:
            nii = nib.load(p)
            images.append(nii.get_fdata(dtype=np.float32))
            if spacing is None:
                zooms = tuple(float(v) for v in nii.header.get_zooms()[:3])
                spatial_unit = nii.header.get_xyzt_units()[0]
                # MONAI's Euclidean HD95 accepts physical spacing. Normalize known NIfTI
                # spatial units to millimetres; when units are unspecified, use voxel units.
                scale_to_mm = {"meter": 1000.0, "mm": 1.0, "micron": 0.001}.get(spatial_unit)
                if scale_to_mm is not None:
                    spacing = tuple(v * scale_to_mm for v in zooms)
                    spacing_unit = "mm"
                else:
                    spacing = (1.0, 1.0, 1.0)
                    spacing_unit = "voxel"
        d["image"] = np.stack(images)
        d["spacing"] = np.asarray(spacing, dtype=np.float32)
        d["spacing_unit"] = spacing_unit
        d["label"] = np.asarray(nib.load(d["label"]).dataobj).astype(np.int16)[None]
        return d


class ToRegions(_TfBase):
    """Label map -> 3 binary channels [ET, TC, WT]. Works for BraTS 2018-2021
    (ET=4) and 2023 (ET=3) via --et_label; TC = NCR(1) + ET; WT = any tumor."""

    def __init__(self, et_label=4):
        self.et_label = et_label

    def __call__(self, data):
        d = dict(data)
        lab = d["label"][0]
        et = lab == self.et_label
        tc = (lab == 1) | et
        wt = lab > 0
        d["label"] = np.stack([et, tc, wt]).astype(np.float32)
        return d


def det_transforms(a):
    import torch
    from monai.transforms import CropForegroundd, EnsureTyped, NormalizeIntensityd
    return [
        LoadBraTS(),
        ToRegions(a.et_label),
        CropForegroundd(keys=["image", "label"], source_key="image", allow_smaller=True),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image", "label"], dtype=torch.float32, track_meta=False),
    ]


def train_random_transforms(a):
    from monai.transforms import (RandCropByPosNegLabeld, RandFlipd,
                                  RandScaleIntensityd, RandShiftIntensityd, SpatialPadd)
    roi = [a.roi] * 3
    return [
        SpatialPadd(keys=["image", "label"], spatial_size=roi),
        RandCropByPosNegLabeld(keys=["image", "label"], label_key="label",
                               spatial_size=roi, pos=1, neg=1,
                               num_samples=a.num_samples, image_key="image",
                               image_threshold=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
    ]


# --------------------------------------------------------------------------- #
# Model / helpers
# --------------------------------------------------------------------------- #
def load_ckpt(path, device):
    import torch
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:                    # older torch without weights_only
        return torch.load(path, map_location=device)


def subsample_cases(cases, max_cases, seed):
    """Deterministic random subset (0 = use all). Applied before the split is created."""
    if not max_cases or max_cases >= len(cases):
        return cases
    idx = list(range(len(cases)))
    random.Random(seed).shuffle(idx)
    return [cases[i] for i in sorted(idx[:max_cases])]


def make_model(device):
    import torch
    from monai.networks.nets import SegResNet
    model = SegResNet(blocks_down=[1, 2, 2, 4], blocks_up=[1, 1, 1], init_filters=16,
                      in_channels=4, out_channels=3, dropout_prob=0.2).to(device)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)      # uses both T4s
    return model


def raw_state_dict(model):
    return (model.module if hasattr(model, "module") else model).state_dict()


def load_state(model, sd):
    (model.module if hasattr(model, "module") else model).load_state_dict(sd)


def mask_vector(mask, device):
    import torch
    return torch.tensor([(mask >> i) & 1 for i in range(4)], dtype=torch.float32,
                        device=device).view(1, 4, 1, 1, 1)


def sample_batch_masks(b, p_full, device):
    """Modality dropout: full input with prob p_full, otherwise a uniformly random
    non-empty subset (at least one sequence is always kept)."""
    import torch
    rows = []
    for _ in range(b):
        m = 15 if random.random() < p_full else random.randint(1, 15)
        rows.append([(m >> i) & 1 for i in range(4)])
    return torch.tensor(rows, dtype=torch.float32, device=device).view(b, 4, 1, 1, 1)


def predict_prob(model, x, a):
    """Return calibrated-input probabilities (sigmoid outputs), before thresholding."""
    import torch
    from monai.inferers import sliding_window_inference
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16,
                                         enabled=a.amp and x.is_cuda):
        logits = sliding_window_inference(x, [a.roi] * 3, a.sw_batch, model,
                                          overlap=0.5, mode="gaussian")
    return torch.sigmoid(logits.float())


def predict(model, x, a):
    """Binary predictions used by validation; threshold matches the evaluation protocol."""
    return (predict_prob(model, x, a) > 0.5).float()


def region_dice(pred, y):
    """Dice per region for a single case. Both-empty -> 1.0 (BraTS convention),
    one-empty -> 0.0. Also returns voxel counts so empty-GT cases can be filtered."""
    out = []
    for r in range(3):
        p, g = pred[0, r], y[0, r]
        sp, sg = float(p.sum()), float(g.sum())
        d = 1.0 if sp + sg == 0 else 2.0 * float((p * g).sum()) / (sp + sg)
        out.append((d, sg, sp))
    return out


def seed_all(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def apply_smoke(a, split):
    if a.smoke:
        a.epochs, a.roi, a.val_interval = 2, 64, 1
        split["train"], split["val"], split["test"] = (
            split["train"][:6], split["val"][:2], split["test"][:2])
    return split


# --------------------------------------------------------------------------- #
# Train
# --------------------------------------------------------------------------- #
def cmd_train(a):
    import torch
    from monai.data import CacheDataset, DataLoader, Dataset
    from monai.losses import DiceLoss
    from monai.transforms import Compose

    seed_all(a.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path(a.out_dir) / a.mode
    out.mkdir(parents=True, exist_ok=True)

    cases = require_cases(a.data_root, a.max_cases, a.split_seed)
    split = apply_smoke(a, get_split(cases, a.out_dir, a.split_seed))
    print(f"cases={len(cases)} train={len(split['train'])} val={len(split['val'])} "
          f"test={len(split['test'])} gpus={torch.cuda.device_count()}")

    det = det_transforms(a)
    train_ds = CacheDataset(split["train"], Compose(det + train_random_transforms(a)),
                            cache_rate=a.cache_rate, num_workers=a.workers)
    train_ld = DataLoader(train_ds, batch_size=a.batch_size, shuffle=True,
                          num_workers=a.workers, pin_memory=True)
    val_ds = Dataset(split["val"], Compose(det))
    val_ld = DataLoader(val_ds, batch_size=1, num_workers=2)

    model = make_model(device)
    loss_fn = DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0, smooth_dr=1e-5)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.epochs)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=a.amp)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=a.amp)

    start, best = 0, -1.0
    if a.resume and (out / "last.pt").exists():
        ck = load_ckpt(out / "last.pt", device)
        load_state(model, ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        start, best = ck["epoch"] + 1, ck["best"]
        print(f"resumed from epoch {start}")

    # validation subsets used for checkpoint selection
    val_masks = [15] if a.mode == "full" else [15, 0b1101, 0b1000, 0b0010]
    log_path = out / "log.csv"
    if not log_path.exists():
        log_path.write_text("epoch,loss,lr,val_score,seconds\n")

    for epoch in range(start, a.epochs):
        t0 = time.time()
        model.train()
        tot, n = 0.0, 0
        for batch in train_ld:
            x = batch["image"].to(device, non_blocking=True)
            y = batch["label"].to(device, non_blocking=True)
            if a.mode == "dropout":
                x = x * sample_batch_masks(x.shape[0], a.p_full, device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16,
                                enabled=a.amp and device.type == "cuda"):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item()
            n += 1
        sched.step()

        score = float("nan")
        if (epoch + 1) % a.val_interval == 0 or epoch + 1 == a.epochs:
            model.eval()
            vals = []
            for vb in val_ld:
                xv, yv = vb["image"].to(device), vb["label"].to(device)
                for m in val_masks:
                    pr = predict(model, xv * mask_vector(m, device), a)
                    vals.append(np.mean([d for d, _, _ in region_dice(pr, yv)]))
            score = float(np.mean(vals))
            if score > best:
                best = score
                torch.save({"model": raw_state_dict(model), "epoch": epoch,
                            "score": score}, out / "best.pt")
        torch.save({"model": raw_state_dict(model), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "epoch": epoch, "best": best},
                   out / "last.pt")
        with open(log_path, "a") as f:
            f.write(f"{epoch},{tot / max(n, 1):.5f},{sched.get_last_lr()[0]:.2e},"
                    f"{score:.4f},{time.time() - t0:.0f}\n")
        print(f"[{a.mode}] epoch {epoch + 1}/{a.epochs} loss={tot / max(n, 1):.4f} "
              f"val={score:.4f} best={best:.4f} ({time.time() - t0:.0f}s)")
    json.dump(vars(a), open(out / "config.json", "w"), indent=1, default=str)


# --------------------------------------------------------------------------- #
# Probability calibration helpers
# --------------------------------------------------------------------------- #
CALIBRATION_BINS = 15

def calibration_sample_plan(y, seed, max_per_class=10000):
    """Build one deterministic voxel sample plan per region, reused for all subsets.

    Sampling is stratified by target class so rare foreground voxels are represented.
    Population-prevalence weights yield ordinary voxel ECE/Brier estimates; equal
    class weights yield class-balanced ECE/Brier estimates. Only indices are stored.
    """
    plans = []
    for r in range(y.shape[1]):
        gt = y[0, r].detach().cpu().numpy().reshape(-1).astype(bool, copy=False)
        n_total = int(gt.size)
        pos_all = np.flatnonzero(gt)
        n_pos = int(pos_all.size)
        n_neg = n_total - n_pos
        rng = np.random.default_rng(seed + r * 9176)
        if n_pos > max_per_class:
            pos_idx = rng.choice(pos_all, size=max_per_class, replace=False)
        else:
            pos_idx = pos_all
        if n_neg == 0:
            neg_idx = np.empty(0, dtype=np.int64)
        elif n_neg <= max_per_class:
            neg_idx = np.flatnonzero(~gt)
        else:
            # Uniform candidate sampling avoids materializing an enormous negative-index array.
            candidate = rng.choice(n_total, size=min(n_total, max_per_class * 4), replace=False)
            candidate = candidate[~gt[candidate]]
            if candidate.size < max_per_class:
                neg_all = np.flatnonzero(~gt)
                neg_idx = rng.choice(neg_all, size=max_per_class, replace=False)
            else:
                neg_idx = rng.choice(candidate, size=max_per_class, replace=False)
        plans.append({"pos_idx": np.asarray(pos_idx, dtype=np.int64),
                      "neg_idx": np.asarray(neg_idx, dtype=np.int64),
                      "n_pos": n_pos, "n_neg": n_neg, "n_total": n_total})
    return plans


def _weighted_probability_ece(prob, target, weights, n_bins=CALIBRATION_BINS):
    if prob.size == 0 or weights.size != prob.size or float(weights.sum()) <= 0:
        return np.nan
    weights = weights / weights.sum()
    bins = np.minimum((np.clip(prob, 0.0, 1.0) * n_bins).astype(int), n_bins - 1)
    value = 0.0
    for b in range(n_bins):
        sel = bins == b
        mass = float(weights[sel].sum())
        if mass <= 0:
            continue
        mean_prob = float(np.sum(weights[sel] * prob[sel]) / mass)
        mean_target = float(np.sum(weights[sel] * target[sel]) / mass)
        value += mass * abs(mean_prob - mean_target)
    return float(value)


def calibration_metrics_from_samples(pos_prob, neg_prob, n_pos, n_neg, n_total):
    """ECE and Brier score, plus class-balanced variants, for one binary region."""
    pos_prob = np.asarray(pos_prob, dtype=np.float64).reshape(-1)
    neg_prob = np.asarray(neg_prob, dtype=np.float64).reshape(-1)
    p = np.concatenate([pos_prob, neg_prob])
    y = np.concatenate([np.ones(pos_prob.size), np.zeros(neg_prob.size)])
    if p.size == 0:
        return {"ece": np.nan, "ece_balanced": np.nan,
                "brier": np.nan, "brier_balanced": np.nan}

    # Weight samples back to their target-voxel class prevalence.
    w_parts = []
    if pos_prob.size:
        w_parts.append(np.full(pos_prob.size, (n_pos / max(n_total, 1)) / pos_prob.size))
    if neg_prob.size:
        w_parts.append(np.full(neg_prob.size, (n_neg / max(n_total, 1)) / neg_prob.size))
    w = np.concatenate(w_parts) if w_parts else np.ones(p.size)
    ece = _weighted_probability_ece(p, y, w)
    brier = float(np.sum((w / w.sum()) * (p - y) ** 2))

    if pos_prob.size and neg_prob.size and n_pos > 0 and n_neg > 0:
        wb = np.concatenate([np.full(pos_prob.size, 0.5 / pos_prob.size),
                             np.full(neg_prob.size, 0.5 / neg_prob.size)])
        ece_bal = _weighted_probability_ece(p, y, wb)
        brier_bal = float(np.sum(wb * (p - y) ** 2))
    else:
        ece_bal, brier_bal = np.nan, np.nan
    return {"ece": ece, "ece_balanced": ece_bal,
            "brier": brier, "brier_balanced": brier_bal}


# --------------------------------------------------------------------------- #
# Evaluate: the 15-subset sweep
# --------------------------------------------------------------------------- #
def cmd_evaluate(a):
    import torch
    from monai.data import DataLoader, Dataset
    from monai.transforms import Compose

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = a.ckpt or str(Path(a.out_dir) / a.mode / "best.pt")
    if not os.path.exists(ckpt):
        sys.exit(f"Checkpoint not found: {ckpt}. Run `train --mode {a.mode}` first "
                 "(and make sure it finished without errors).")
    cases = require_cases(a.data_root, a.max_cases, a.split_seed)
    split = apply_smoke(a, get_split(cases, a.out_dir, a.split_seed))
    ds = Dataset(split["test"], Compose(det_transforms(a)))
    ld = DataLoader(ds, batch_size=1, num_workers=2)

    model = make_model(device)
    load_state(model, load_ckpt(ckpt, device)["model"])
    model.eval()

    hd95_fn = None
    if a.hd95:
        from monai.metrics import compute_hausdorff_distance as hd95_fn

    rows = []
    for i, (batch, case) in enumerate(zip(ld, split["test"])):
        x, y = batch["image"].to(device), batch["label"].to(device)
        calib_plans = calibration_sample_plan(
            y, seed=int(a.seed + i * 10007), max_per_class=a.calib_samples_per_class)
        spacing_val = batch.get("spacing")
        if spacing_val is not None:
            if hasattr(spacing_val, "detach"):
                spacing_val = spacing_val[0].detach().cpu().numpy().reshape(-1).tolist()
            elif isinstance(spacing_val, (list, tuple)) and len(spacing_val) and hasattr(spacing_val[0], "tolist"):
                spacing_val = spacing_val[0].tolist()
            spacing = tuple(float(v) for v in spacing_val)
        else:
            spacing = None
        spacing_unit_val = batch.get("spacing_unit", "voxel")
        while isinstance(spacing_unit_val, (list, tuple)) and len(spacing_unit_val) == 1:
            spacing_unit_val = spacing_unit_val[0]
        spacing_unit = str(spacing_unit_val)
        for m in ALL_MASKS:
            probs = predict_prob(model, x * mask_vector(m, device), a)
            pr = (probs > 0.5).float()
            dd = region_dice(pr, y)
            row = {"case": case["id"], "mask": m, "subset": subset_name(m),
                   "k": int(POPCOUNT[m]), "hd95_unit": spacing_unit if a.hd95 else "not_computed"}
            for r, name in enumerate(REGIONS):
                row[f"dice_{name}"] = dd[r][0]
                row[f"gt_vox_{name}"] = dd[r][1]
                row[f"pred_vox_{name}"] = dd[r][2]
                row[f"gt_empty_{name}"] = dd[r][1] == 0
                row[f"hd95_{name}"] = np.nan

                plan = calib_plans[r]
                pflat = probs[0, r].reshape(-1)
                pos_idx = torch.as_tensor(plan["pos_idx"], dtype=torch.long, device=pflat.device)
                neg_idx = torch.as_tensor(plan["neg_idx"], dtype=torch.long, device=pflat.device)
                p_pos = pflat[pos_idx].detach().float().cpu().numpy()
                p_neg = pflat[neg_idx].detach().float().cpu().numpy()
                cm = calibration_metrics_from_samples(
                    p_pos, p_neg, plan["n_pos"], plan["n_neg"], plan["n_total"])
                for metric, value in cm.items():
                    row[f"{metric}_{name}"] = value

            if hd95_fn is not None:
                kwargs = {"include_background": True, "percentile": 95}
                if spacing is not None:
                    kwargs["spacing"] = spacing
                h = hd95_fn(pr, y, **kwargs)[0]
                for r, name in enumerate(REGIONS):
                    v = float(h[r])
                    row[f"hd95_{name}"] = v if not np.isnan(v) else np.nan
            rows.append(row)
        print(f"[{a.mode}] evaluated case {i + 1}/{len(split['test'])}: {case['id']}")
    out_csv = Path(a.out_dir) / f"results_{a.mode}.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    metadata = {"mode": a.mode, "seed": a.seed, "split_seed": a.split_seed,
                "calibration_bins": CALIBRATION_BINS,
                "calibration_samples_per_class_max": a.calib_samples_per_class,
                "calibration_definition": "voxel probability reliability ECE; class-balanced variant weights positive and negative classes equally",
                "brier_definition": "weighted voxel-wise squared probability error",
                "hd95_computed": bool(a.hd95),
                "hd95_units": sorted(set(pd.DataFrame(rows)["hd95_unit"].astype(str))) if a.hd95 else [],
                "hd95_unit_rule": "known NIfTI spatial units converted to mm; unspecified units use voxel units"}
    json.dump(metadata, open(Path(a.out_dir) / f"results_{a.mode}_metadata.json", "w"), indent=2)
    print("wrote", out_csv)


# --------------------------------------------------------------------------- #
# Analysis: Shapley, budget curves, minimal sufficient sets
# --------------------------------------------------------------------------- #
def build_array(df, metric="dice", policy="convention"):
    """-> array [cases, 16, 3]; index 0 (empty set) stays NaN and is set to 0 later."""
    cases = sorted(df["case"].unique())
    cidx = {c: i for i, c in enumerate(cases)}
    arr = np.full((len(cases), 16, 3), np.nan)
    for r, name in enumerate(REGIONS):
        vals = df[f"{metric}_{name}"].to_numpy(dtype=float)
        if policy == "exclude":
            vals = np.where(df[f"gt_empty_{name}"].to_numpy(dtype=bool), np.nan, vals)
        arr[[cidx[c] for c in df["case"]], df["mask"].to_numpy(), r] = vals
    return cases, arr


def subset_values(arr, idx=None):
    """Mean over (resampled) cases -> v[..., 16, 3] with v(empty set) = 0."""
    a = arr if idx is None else arr[idx]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)   # empty set = all-NaN slice
        v = np.nanmean(a, axis=-3)
    v[..., 0, :] = 0.0
    return v


def shapley_values(v):
    """Exact Shapley value of each of the 4 sequences. v: [..., 16, R] -> [..., 4, R]."""
    n = 4
    out = np.zeros(v.shape[:-2] + (n, v.shape[-1]))
    for m in range(n):
        for S in range(16):
            if S & (1 << m):
                continue
            s = int(POPCOUNT[S])
            w = math.factorial(s) * math.factorial(n - s - 1) / math.factorial(n)
            out[..., m, :] += w * (v[..., S | (1 << m), :] - v[..., S, :])
    return out


def budget_stats(v):
    """Best / mean / worst subset value at each budget size k=1..4 -> [..., 3, 4, R]."""
    res = []
    for k in range(1, 5):
        sel = [m for m in ALL_MASKS if POPCOUNT[m] == k]
        sub = v[..., sel, :]
        res.append(np.stack([sub.max(-2), sub.mean(-2), sub.min(-2)], axis=-2))
    return np.stack(res, axis=-2)       # [..., stat(3), k(4), R]  after swap below


def minimal_sufficient(v, costs, eps):
    """Lowest-cost subset S with v(S) >= (1-eps) v(full); ties -> fewer sequences,
    then higher value. Returns mask index [..., R]."""
    cost = np.array([sum(costs[MODS[i]] for i in range(4) if (m >> i) & 1)
                     for m in range(16)])
    ok = v >= (1 - eps) * v[..., 15:16, :]
    ok[..., 0, :] = False
    key = cost[:, None] * 1e4 + POPCOUNT[:, None] * 10.0 - v
    key = np.where(ok, key, np.inf)
    return key.argmin(axis=-2), cost


def fmt_ci(p, lo, hi):
    return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"


def analyze_one(df, tag, out, a, costs):
    cases, arr = build_array(df, "dice", a.empty_policy)
    n = len(cases)
    rng = np.random.default_rng(a.seed)
    idx = rng.integers(0, n, size=(a.n_boot, n))

    v = subset_values(arr)                                   # point estimate [16, R]
    vb = np.stack([subset_values(arr, idx[b]) for b in range(a.n_boot)])  # [B,16,R]
    phi, phib = shapley_values(v), shapley_values(vb)
    lo = lambda x: np.percentile(x, 2.5, axis=0)
    hi = lambda x: np.percentile(x, 97.5, axis=0)

    # --- subset table
    rows = []
    for m in ALL_MASKS:
        row = {"model": tag, "subset": subset_name(m), "k": int(POPCOUNT[m])}
        for r, name in enumerate(REGIONS):
            row[f"dice_{name}"] = v[m, r]
            row[f"dice_{name}_lo"] = lo(vb)[m, r]
            row[f"dice_{name}_hi"] = hi(vb)[m, r]
        rows.append(row)
    sub_df = pd.DataFrame(rows)
    sub_df.to_csv(out / f"subset_table_{tag}.csv", index=False)

    # --- Shapley
    rows = []
    for m in range(4):
        for r, name in enumerate(REGIONS):
            rows.append({"model": tag, "sequence": MODS[m], "region": name,
                         "phi": phi[m, r], "phi_lo": lo(phib)[m, r],
                         "phi_hi": hi(phib)[m, r],
                         "share_of_full": phi[m, r] / v[15, r] if v[15, r] else np.nan})
    sh_df = pd.DataFrame(rows)
    sh_df.to_csv(out / f"shapley_{tag}.csv", index=False)
    eff = np.abs(phi.sum(0) - v[15]).max()                   # efficiency axiom check
    assert eff < 1e-9, f"Shapley efficiency check failed: {eff}"

    # --- budget curves (stats axis: 0 best, 1 mean, 2 worst)
    bs, bsb = budget_stats(v), budget_stats(vb)              # [3,4,R] / [B,3,4,R]
    rows = []
    for k in range(4):
        for si, sname in enumerate(["best", "mean", "worst"]):
            for r, name in enumerate(REGIONS):
                rows.append({"model": tag, "k": k + 1, "stat": sname, "region": name,
                             "dice": bs[si, k, r], "lo": lo(bsb)[si, k, r],
                             "hi": hi(bsb)[si, k, r]})
    bc_df = pd.DataFrame(rows)
    bc_df.to_csv(out / f"budget_curve_{tag}.csv", index=False)

    # --- minimal sufficient sets with bootstrap stability
    rows = []
    for eps in a.eps:
        pick, cost = minimal_sufficient(v, costs, eps)
        pickb, _ = minimal_sufficient(vb, costs, eps)
        for r, name in enumerate(REGIONS):
            m = int(pick[r])
            rows.append({"model": tag, "region": name, "eps": eps,
                         "min_sufficient_set": subset_name(m), "cost": cost[m],
                         "full_cost": cost[15], "cost_saving": 1 - cost[m] / cost[15],
                         "dice": v[m, r], "dice_full": v[15, r],
                         "selection_stability": float((pickb[:, r] == m).mean())})
    ms_df = pd.DataFrame(rows)
    ms_df.to_csv(out / f"minimal_sets_{tag}.csv", index=False)

    # --- LaTeX table of Shapley values
    lines = ["\\begin{tabular}{lccc}", "\\hline",
             "Sequence & ET & TC & WT \\\\", "\\hline"]
    for m in range(4):
        cells = [fmt_ci(phi[m, r], lo(phib)[m, r], hi(phib)[m, r]) for r in range(3)]
        lines.append(f"{MODS[m]} & " + " & ".join(cells) + " \\\\")
    lines += ["\\hline", "Full input (sum) & " +
              " & ".join(f"{v[15, r]:.3f}" for r in range(3)) + " \\\\",
              "\\hline", "\\end{tabular}"]
    (out / f"shapley_{tag}.tex").write_text("\n".join(lines))

    print(f"\n=== {tag}: {n} test cases, empty-GT policy = {a.empty_policy} ===")
    print(sh_df.pivot(index="sequence", columns="region", values="phi")
          .reindex(MODS)[REGIONS].round(3))
    print(ms_df[ms_df.eps == a.eps[len(a.eps) // 2]][
        ["region", "eps", "min_sufficient_set", "cost_saving", "dice", "dice_full",
         "selection_stability"]].round(3).to_string(index=False))
    return {"v": v, "vb": vb, "phi": phi, "phib": phib, "bs": bs, "bsb": bsb,
            "sub": sub_df, "n": n}


def make_figures(res, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    colors = {"ET": "#c0392b", "TC": "#2874a6", "WT": "#1e8449"}
    # Fig 1: Shapley bars (one panel per model)
    for tag, d in res.items():
        fig, ax = plt.subplots(figsize=(5.2, 3.2))
        w = 0.26
        lo = np.percentile(d["phib"], 2.5, axis=0)
        hi = np.percentile(d["phib"], 97.5, axis=0)
        for r, name in enumerate(REGIONS):
            xs = np.arange(4) + (r - 1) * w
            ax.bar(xs, d["phi"][:, r], w, color=colors[name], label=name,
                   yerr=[d["phi"][:, r] - lo[:, r], hi[:, r] - d["phi"][:, r]],
                   capsize=2, linewidth=0)
        ax.set_xticks(range(4), MODS)
        ax.set_ylabel("Shapley marginal Dice")
        ax.set_title(f"Per-sequence marginal utility ({tag})", fontsize=9)
        ax.legend(frameon=False, fontsize=8)
        fig.tight_layout()
        fig.savefig(out / f"fig_shapley_{tag}.png", dpi=300)
        plt.close(fig)

        # Fig 3: subset x region heat map
        order = sorted(ALL_MASKS, key=lambda m: (POPCOUNT[m], m))
        fig, ax = plt.subplots(figsize=(3.6, 5.2))
        mat = d["v"][order]
        im = ax.imshow(mat, cmap="viridis", vmin=0, vmax=max(0.95, mat.max()), aspect="auto")
        ax.set_xticks(range(3), REGIONS)
        ax.set_yticks(range(15), [subset_name(m) for m in order], fontsize=7)
        for i in range(15):
            for j in range(3):
                ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                        fontsize=6.5, color="w" if mat[i, j] < 0.6 else "k")
        ax.set_title(f"Dice by available subset ({tag})", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.05)
        fig.tight_layout()
        fig.savefig(out / f"fig_subset_heatmap_{tag}.png", dpi=300)
        plt.close(fig)

    # Fig 2: budget curves, models overlaid
    fig, axes = plt.subplots(1, 3, figsize=(8.2, 2.8), sharey=True)
    styles = itertools.cycle(["-", "--", ":"])
    for tag, d in res.items():
        ls = next(styles)
        blo = np.percentile(d["bsb"], 2.5, axis=0)
        bhi = np.percentile(d["bsb"], 97.5, axis=0)
        for r, name in enumerate(REGIONS):
            ax = axes[r]
            ax.plot(range(1, 5), d["bs"][0, :, r], ls, color=colors[name],
                    marker="o", ms=3, label=f"{tag} best")
            ax.plot(range(1, 5), d["bs"][1, :, r], ls, color="gray",
                    marker="s", ms=3, label=f"{tag} mean")
            ax.fill_between(range(1, 5), blo[1, :, r], bhi[1, :, r], color="gray", alpha=0.15)
            ax.set_title(name, fontsize=9)
            ax.set_xlabel("# sequences available (k)")
            ax.set_xticks(range(1, 5))
    axes[0].set_ylabel("Dice")
    axes[0].legend(frameon=False, fontsize=6)
    fig.tight_layout()
    fig.savefig(out / "fig_budget_curves.png", dpi=300)
    plt.close(fig)


# --------------------------------------------------------------------------- #
# Safety-metric aggregation: boundary distance and calibration
# --------------------------------------------------------------------------- #
def _bootstrap_nanmean_ci(values, n_boot, rng):
    x = np.asarray(values, dtype=float)
    finite = np.isfinite(x)
    if not finite.any():
        return np.nan, np.nan
    # Infinite HD95 means exactly one of prediction/ground truth is empty; do not
    # mix it into the conditional mean. Its frequency is separately reported below.
    xx = np.where(finite, x, np.nan)
    idx = rng.integers(0, len(xx), size=(n_boot, len(xx)))
    sampled = xx[idx]
    denom = np.isfinite(sampled).sum(axis=1)
    nums = np.nansum(sampled, axis=1)
    means = np.divide(nums, denom, out=np.full(n_boot, np.nan), where=denom > 0)
    means = means[np.isfinite(means)]
    if means.size == 0:
        return np.nan, np.nan
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def summarize_safety_metrics(df, tag, out, a):
    metric_prefixes = [
        ("hd95", "HD95"),
        ("ece", "ECE_voxel"),
        ("ece_balanced", "ECE_class_balanced"),
        ("brier", "Brier_voxel"),
        ("brier_balanced", "Brier_class_balanced"),
    ]
    available = [(p, label) for p, label in metric_prefixes
                 if any(f"{p}_{r}" in df.columns for r in REGIONS)]
    rows = []
    for mask in ALL_MASKS:
        dmask = df[df["mask"] == mask]
        for region in REGIONS:
            for prefix, label in available:
                col = f"{prefix}_{region}"
                if col not in dmask.columns:
                    continue
                per_case = dmask.groupby("case", sort=False)[col].mean()
                values = per_case.to_numpy(dtype=float)
                good = values[np.isfinite(values)]
                rng = np.random.default_rng(a.seed + mask * 101 + REGIONS.index(region) * 13 + len(label))
                lo, hi = _bootstrap_nanmean_ci(values, a.n_boot, rng)
                metric_label = label
                if prefix == "hd95":
                    units = sorted(dmask["hd95_unit"].dropna().astype(str).unique()) if "hd95_unit" in dmask.columns else []
                    metric_label = f"HD95_{'_'.join(units)}" if units else "HD95_unit_unknown"
                rows.append({"model": tag, "mask": mask, "subset": subset_name(mask),
                             "k": int(POPCOUNT[mask]), "region": region, "metric": metric_label,
                             "mean": float(np.mean(good)) if good.size else np.nan,
                             "median": float(np.median(good)) if good.size else np.nan,
                             "ci95_lo": lo, "ci95_hi": hi,
                             "n_defined_finite": int(good.size),
                             "n_infinite": int(np.isinf(values).sum()),
                             "n_nan_or_undefined": int(np.isnan(values).sum()),
                             "n_total": int(values.size)})
    if rows:
        pd.DataFrame(rows).to_csv(out / f"safety_metrics_{tag}.csv", index=False)


# --------------------------------------------------------------------------- #
# Cost sensitivity and cost-free minimum-sequence analysis
# --------------------------------------------------------------------------- #
def write_cost_sensitivity(res, base_costs, eps_values, t1ce_grid, out):
    sweep_rows, free_rows = [], []
    equal_costs = {m: 1.0 for m in MODS}
    for tag, d in res.items():
        v = d["v"]
        for eps in eps_values:
            picks_free, costs_free = minimal_sufficient(v, equal_costs, eps)
            picks_free_boot, _ = minimal_sufficient(d["vb"], equal_costs, eps)
            for r, region in enumerate(REGIONS):
                mm = int(picks_free[r])
                free_rows.append({"model": tag, "region": region, "eps": eps,
                                  "min_sequence_set": subset_name(mm),
                                  "n_sequences": int(POPCOUNT[mm]),
                                  "equal_unit_cost": float(costs_free[mm]),
                                  "dice": float(v[mm, r]), "dice_full": float(v[15, r]),
                                  "selection_stability": float(np.mean(picks_free_boot[:, r] == mm))})
            for t1ce_cost in t1ce_grid:
                costs = dict(base_costs)
                costs["T1ce"] = float(t1ce_cost)
                picks, total_cost = minimal_sufficient(v, costs, eps)
                picks_boot, _ = minimal_sufficient(d["vb"], costs, eps)
                for r, region in enumerate(REGIONS):
                    mm = int(picks[r])
                    sweep_rows.append({"model": tag, "region": region, "eps": eps,
                                       "t1ce_cost": float(t1ce_cost),
                                       "selected_set": subset_name(mm),
                                       "n_sequences": int(POPCOUNT[mm]),
                                       "selected_cost": float(total_cost[mm]),
                                       "full_cost": float(total_cost[15]),
                                       "cost_saving": float(1 - total_cost[mm] / total_cost[15]),
                                       "selection_stability": float(np.mean(picks_boot[:, r] == mm)),
                                       "dice": float(v[mm, r]), "dice_full": float(v[15, r])})
    if sweep_rows:
        sweep = pd.DataFrame(sweep_rows).sort_values(["model", "region", "eps", "t1ce_cost"])
        sweep.to_csv(out / "cost_sensitivity.csv", index=False)
        changes = []
        for keys, g in sweep.groupby(["model", "region", "eps"], sort=False):
            previous = None
            for row in g.to_dict("records"):
                if row["selected_set"] != previous:
                    changes.append({"model": row["model"], "region": row["region"],
                                    "eps": row["eps"], "t1ce_cost": row["t1ce_cost"],
                                    "selected_set": row["selected_set"],
                                    "selected_cost": row["selected_cost"],
                                    "selection_stability": row["selection_stability"],
                                    "dice": row["dice"]})
                    previous = row["selected_set"]
        pd.DataFrame(changes).to_csv(out / "cost_sensitivity_transitions.csv", index=False)
    if free_rows:
        pd.DataFrame(free_rows).to_csv(out / "cost_free_minimal_sets.csv", index=False)


# --------------------------------------------------------------------------- #
# Paired case-level inference: modality dropout versus full-input model
# --------------------------------------------------------------------------- #
def _holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full(p.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    if not len(valid):
        return out
    order_local = np.argsort(p[valid])
    ordered_idx = valid[order_local]
    m = len(ordered_idx)
    adjusted = np.maximum.accumulate((m - np.arange(m)) * p[ordered_idx])
    adjusted = np.minimum(adjusted, 1.0)
    out[ordered_idx] = adjusted
    return out


def paired_case_inference(dropout_df, full_df, out, a):
    keys = ["case", "mask", "subset", "k"]
    d = dropout_df[keys + [f"dice_{r}" for r in REGIONS]].copy()
    f = full_df[keys + [f"dice_{r}" for r in REGIONS]].copy()
    merged = d.merge(f, on=keys, suffixes=("_dropout", "_full"), validate="one_to_one")
    if merged.empty:
        warnings.warn("Paired comparison skipped: no matching case/subset rows.")
        return
    rng = np.random.default_rng(a.seed + 5047)
    rows = []

    def add_endpoint(label, region, diff, mask=None, subset=None, k=None):
        diff = np.asarray(diff, dtype=float)
        diff = diff[np.isfinite(diff)]
        n = int(diff.size)
        if not n:
            return
        idx = rng.integers(0, n, size=(a.n_boot, n))
        boots = diff[idx].mean(axis=1)
        estimate = float(diff.mean())
        signs = rng.choice(np.array([-1, 1], dtype=np.int8), size=(a.n_perm, n), replace=True)
        permuted = (signs * diff[None, :]).mean(axis=1)
        p = float((np.count_nonzero(np.abs(permuted) >= abs(estimate)) + 1) / (a.n_perm + 1))
        rows.append({"endpoint": label, "region": region, "mask": mask,
                     "subset": subset, "k": k, "n_cases": n,
                     "mean_difference_dropout_minus_full": estimate,
                     "ci95_lo": float(np.percentile(boots, 2.5)),
                     "ci95_hi": float(np.percentile(boots, 97.5)),
                     "paired_sign_flip_p": p})

    for mask in ALL_MASKS:
        dm = merged[merged["mask"] == mask]
        if dm.empty:
            continue
        for region in REGIONS:
            add_endpoint("subset", region,
                         dm[f"dice_{region}_dropout"].to_numpy() - dm[f"dice_{region}_full"].to_numpy(),
                         mask=mask, subset=subset_name(mask), k=int(POPCOUNT[mask]))

    # Paper's k=1 headline: average the four singleton-input scores within each case,
    # then compare those per-case means (do not treat modalities as independent cases).
    for region in REGIONS:
        one = merged[merged["k"] == 1]
        if not one.empty:
            ag = one.groupby("case")[[f"dice_{region}_dropout", f"dice_{region}_full"]].mean()
            add_endpoint("mean_single_sequence", region,
                         ag[f"dice_{region}_dropout"].to_numpy() - ag[f"dice_{region}_full"].to_numpy(),
                         mask=None, subset="mean of four singleton inputs", k=1)

    if rows:
        result = pd.DataFrame(rows)
        result["holm_adjusted_p"] = _holm_adjust(result["paired_sign_flip_p"].to_numpy())
        result["test_note"] = "Two-sided paired case-level sign-flip permutation; percentile paired bootstrap CI; Holm correction over rows."
        result.to_csv(out / "paired_bootstrap_comparison.csv", index=False)


def cmd_analyze(a):
    costs = DEFAULT_COSTS
    if a.costs:
        try:
            costs = json.load(open(a.costs)) if os.path.exists(a.costs) else json.loads(a.costs)
        except Exception as e:
            sys.exit(f"--costs must be JSON text or a path to a JSON file, got {a.costs!r}: {e}")
    missing = [m for m in MODS if m not in costs]
    if missing:
        sys.exit(f"--costs is missing sequences: {missing}")
    out = Path(a.out_dir) / "analysis"
    out.mkdir(parents=True, exist_ok=True)
    res = {}
    input_dfs = {}
    for spec in a.results:
        tag, path = spec.split("=", 1) if "=" in spec else (Path(spec).stem, spec)
        df = pd.read_csv(path)
        input_dfs[tag] = df
        res[tag] = analyze_one(df, tag, out, a, costs)
        summarize_safety_metrics(df, tag, out, a)
    if len(res) > 1:
        side = None
        for tag, d in res.items():
            s = d["sub"].set_index("subset")[[f"dice_{r}" for r in REGIONS]]
            s.columns = [f"{c}_{tag}" for c in s.columns]
            side = s if side is None else side.join(s)
        side.to_csv(out / "compare_subsets.csv")
    write_cost_sensitivity(res, costs, a.eps, a.t1ce_cost_grid, out)
    if "dropout" in input_dfs and "full" in input_dfs:
        paired_case_inference(input_dfs["dropout"], input_dfs["full"], out, a)
    make_figures(res, out)
    json.dump({"costs": costs, "eps": a.eps, "n_boot": a.n_boot,
               "n_perm": a.n_perm, "t1ce_cost_grid": a.t1ce_cost_grid,
               "empty_policy": a.empty_policy}, open(out / "analysis_config.json", "w"))
    print("\nAll outputs written to", out)


def cmd_selftest(a):
    """Synthetic data: ET relies on T1ce, WT on FLAIR/T2. Verifies the analysis pipeline."""
    rng = np.random.default_rng(0)
    w = {"ET": [0.05, 0.60, 0.05, 0.10], "TC": [0.10, 0.40, 0.25, 0.10],
         "WT": [0.05, 0.10, 0.30, 0.45]}
    rows = []
    for c in range(40):
        for m in ALL_MASKS:
            row = {"case": f"case{c:03d}", "mask": m, "subset": subset_name(m),
                   "k": int(POPCOUNT[m])}
            for name in REGIONS:
                s = sum(w[name][i] for i in range(4) if (m >> i) & 1)
                row[f"dice_{name}"] = float(np.clip(0.15 + s + rng.normal(0, 0.05), 0, 1))
                row[f"gt_empty_{name}"] = False
                row[f"hd95_{name}"] = np.nan
            rows.append(row)
    Path(a.out_dir).mkdir(parents=True, exist_ok=True)
    csv = Path(a.out_dir) / "results_synthetic.csv"
    pd.DataFrame(rows).to_csv(csv, index=False)
    a.results = [f"synthetic={csv}"]
    cmd_analyze(a)


# --------------------------------------------------------------------------- #
def build_parser():
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)

    def common(sp):
        sp.add_argument("--data_root", default=DEFAULT_ROOT)
        sp.add_argument("--out_dir", default=DEFAULT_OUT)
        sp.add_argument("--seed", type=int, default=42,
                        help="training/randomness seed; does not change split when --split_seed is fixed")
        sp.add_argument("--split_seed", type=int, default=42,
                        help="seed for selecting cases and creating the fixed train/val/test split")
        sp.add_argument("--roi", type=int, default=128)
        sp.add_argument("--max_cases", type=int, default=0,
                        help="use a random subset of N cases (0 = all). Must be identical "
                             "for train and evaluate; the split is created once per out_dir")
        sp.add_argument("--et_label", type=int, default=4,
                        help="label id of enhancing tumor (4 for BraTS<=2021, 3 for 2023)")
        sp.add_argument("--sw_batch", type=int, default=4)
        sp.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)
        sp.add_argument("--smoke", action="store_true",
                        help="tiny 2-epoch run to test the pipeline end-to-end")

    t = sub.add_parser("train")
    common(t)
    t.add_argument("--mode", choices=["dropout", "full"], default="dropout")
    t.add_argument("--epochs", type=int, default=100)
    t.add_argument("--batch_size", type=int, default=2)
    t.add_argument("--num_samples", type=int, default=2)
    t.add_argument("--lr", type=float, default=2e-4)
    t.add_argument("--p_full", type=float, default=0.25)
    t.add_argument("--val_interval", type=int, default=5)
    t.add_argument("--cache_rate", type=float, default=0.2,
                   help="fraction of training volumes cached in RAM (~80 MB each)")
    t.add_argument("--workers", type=int, default=4)
    t.add_argument("--resume", action="store_true")
    t.set_defaults(fn=cmd_train)

    e = sub.add_parser("evaluate")
    common(e)
    e.add_argument("--mode", choices=["dropout", "full"], default="dropout")
    e.add_argument("--ckpt", default=None)
    e.add_argument("--hd95", action="store_true", help="also compute HD95 in physical units when NIfTI spacing is available (slower)")
    e.add_argument("--calib_samples_per_class", type=int, default=10000,
                   help="maximum foreground/background voxels sampled per case-region for ECE/Brier")
    e.add_argument("--val_interval", type=int, default=5)   # only used by --smoke
    e.add_argument("--epochs", type=int, default=0)
    e.set_defaults(fn=cmd_evaluate)

    def ana(sp):
        sp.add_argument("--out_dir", default=DEFAULT_OUT)
        sp.add_argument("--seed", type=int, default=42)
        sp.add_argument("--n_boot", type=int, default=1000)
        sp.add_argument("--eps", type=float, nargs="+", default=[0.01, 0.02, 0.05, 0.10])
        sp.add_argument("--costs", default=None,
                        help='JSON, e.g. \'{"T1":1,"T1ce":3,"T2":1,"FLAIR":1}\'')
        sp.add_argument("--empty_policy", choices=["convention", "exclude"],
                        default="convention",
                        help="convention: empty GT & empty pred = Dice 1 (BraTS); "
                             "exclude: drop cases whose GT region is empty")
        sp.add_argument("--t1ce_cost_grid", type=float, nargs="+", default=[float(x) for x in range(1, 11)],
                        help="relative T1ce costs for sensitivity sweep; all other sequence costs stay fixed")
        sp.add_argument("--n_perm", type=int, default=10000,
                        help="sign-flip permutations for paired dropout-versus-full comparisons")

    an = sub.add_parser("analyze")
    ana(an)
    an.add_argument("--results", nargs="+", required=True,
                    help="one or more [name=]results.csv (e.g. dropout=... full=...)")
    an.set_defaults(fn=cmd_analyze)

    st = sub.add_parser("selftest")
    ana(st)
    st.set_defaults(fn=cmd_selftest)
    return p


def main(argv=None):
    """Command-line entry point. In a notebook call it with a list, e.g.
    main(["train", "--mode", "dropout", "--smoke"])."""
    args = build_parser().parse_args(argv)
    args.fn(args)


_COMMANDS = ("train", "evaluate", "analyze", "selftest")

if __name__ == "__main__":
    in_notebook = "ipykernel" in sys.modules or "google.colab" in sys.modules
    if in_notebook and not any(a in _COMMANDS for a in sys.argv[1:]):
        # Pasted into a notebook cell: do not parse the kernel's own arguments.
        print("Definitions loaded. In a new cell run, for example:\n"
              '  main(["train", "--mode", "dropout", "--smoke"])\n'
              '  main(["evaluate", "--mode", "dropout", "--smoke"])\n'
              '  main(["analyze", "--results", f"dropout={DEFAULT_OUT}/results_dropout.csv"])')
    else:
        main()


Writing modality_budget_experiment.py


## 3. Settings (edit here only)
- `MAX_CASES`: BraTS 2021 has 1251 cases, which is too many to train two models inside Kaggle's session limit. 400 (≈280 train / 40 val / 80 test) is a sensible start; use 0 for all cases.
- `EPOCHS`: set after you see the seconds-per-epoch in the first real epoch.
- `CACHE_RATE`: fraction of training volumes held in RAM (~80 MB each). Lower it if you run out of memory.

In [3]:
import os, sys, glob, json, shutil, subprocess, time

DATA       = "/kaggle/temp/brats"          # where the tar is extracted
OUT        = "/kaggle/working/mb_exp"      # primary experiment results/checkpoints
SMOKE_OUT  = "/kaggle/working/mb_exp_smoke"  # isolated smoke-test output; never overwrites primary results
MAX_CASES  = 400
EPOCHS     = 50
CACHE_RATE = 0.3
ROI        = 128                           # use 96 if you hit GPU out-of-memory

def run(*args):
    """Run the experiment script as a subprocess and stream its output. No shell involved."""
    cmd = [sys.executable, "modality_budget_experiment.py", *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc} (see output above)")

COMMON = ["--data_root", DATA, "--out_dir", OUT, "--max_cases", MAX_CASES, "--roi", ROI]
print("settings ok")

settings ok


## 4. Extract the data (skipped automatically if already extracted)
Files in `/kaggle/temp` disappear when the session ends, so this cell must be re-run in a new session.

In [4]:
tars = glob.glob("/kaggle/input/**/BraTS2021_Training_Data.tar", recursive=True)
assert tars, "BraTS2021_Training_Data.tar not found under /kaggle/input; check Add Input"
TAR = tars[0]
os.makedirs(DATA, exist_ok=True)
n_seg = len(glob.glob(f"{DATA}/**/*_seg.nii.gz", recursive=True))
if n_seg >= 1251:
    print(f"already extracted: {n_seg} seg files")
else:
    t0 = time.time()
    subprocess.run(["tar", "-xf", TAR, "-C", DATA], check=True)
    print(f"extracted in {time.time()-t0:.0f}s; seg files:", len(glob.glob(f"{DATA}/**/*_seg.nii.gz", recursive=True)))

extracted in 167s; seg files: 1251


## 5. Data check (must show 1251 usable cases and labels {0, 1, 2, 4})

In [5]:
import importlib.util, numpy as np, nibabel as nib
spec = importlib.util.spec_from_file_location("mb", "modality_budget_experiment.py")
mb = importlib.util.module_from_spec(spec); spec.loader.exec_module(mb)
cases = mb.find_cases(DATA)
print("usable cases:", len(cases))
c = cases[0]
print(c["id"], [os.path.basename(p) for p in c["image"]])
img = nib.load(c["image"][1]).shape
lab = np.unique(np.asarray(nib.load(c["label"]).dataobj))
print("volume shape:", img, "| label values:", lab.tolist(), "(BraTS 2021 uses ET = 4, the script default)")

usable cases: 1251
BraTS2021_00000 ['BraTS2021_00000_t1.nii.gz', 'BraTS2021_00000_t1ce.nii.gz', 'BraTS2021_00000_t2.nii.gz', 'BraTS2021_00000_flair.nii.gz']
volume shape: (240, 240, 155) | label values: [0, 1, 2, 4] (BraTS 2021 uses ET = 4, the script default)


## 6. Optional smoke test (a few minutes). The numbers are not scientific results.
Run this only to test a fresh environment. It uses an isolated directory and will not delete the primary experiment/checkpoints.


In [6]:
shutil.rmtree(SMOKE_OUT, ignore_errors=True)
SMOKE_COMMON = ["--data_root", DATA, "--out_dir", SMOKE_OUT, "--max_cases", MAX_CASES, "--roi", ROI]
run("train", "--mode", "dropout", "--smoke", *SMOKE_COMMON)
run("evaluate", "--mode", "dropout", "--smoke", *SMOKE_COMMON)
run("analyze", "--out_dir", SMOKE_OUT, "--n_boot", 200, "--results", f"dropout={SMOKE_OUT}/results_dropout.csv")

$ /usr/bin/python3 modality_budget_experiment.py train --mode dropout --smoke --data_root /kaggle/temp/brats --out_dir /kaggle/working/mb_exp_smoke --max_cases 400 --roi 128
cases=400 train=6 val=2 test=2 gpus=2

Loading dataset: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]
[dropout] epoch 1/2 loss=0.8389 val=0.0660 best=0.0660 (18s)
[dropout] epoch 2/2 loss=0.7563 val=0.0537 best=0.0660 (14s)
$ /usr/bin/python3 modality_budget_experiment.py evaluate --mode dropout --smoke --data_root /kaggle/temp/brats --out_dir /kaggle/working/mb_exp_smoke --max_cases 400 --roi 128
[dropout] evaluated case 1/2: BraTS2021_01216
[dropout] evaluated case 2/2: BraTS2021_00423
wrote /kaggle/working/mb_exp_smoke/results_dropout.csv
$ /usr/bin/python3 modality_budget_experiment.py analyze --out_dir /kaggle/working/mb_exp_smoke --n_boot 200 --results dropout=/kaggle/working/mb_exp_smoke/results_dropout.csv

=== dropout: 2 test cases, empty-GT policy = convention ===
region       ET     TC     WT
sequence   

## 7. Train or resume the modality-dropout model
This cell preserves completed runs and resumes from `last.pt`. It automatically removes only a legacy smoke-test directory if one exists without a full-input checkpoint. To deliberately start a fresh experiment, delete `OUT` yourself or set a separate `OUT` path; do not reset it if you want to reuse existing weights for HD95/ECE. After training, use *Save Version* to preserve outputs/checkpoints.


In [ ]:
from pathlib import Path
legacy_cfg = Path(OUT) / "dropout" / "config.json"
legacy_full = Path(OUT) / "full" / "best.pt"
if legacy_cfg.exists() and not legacy_full.exists():
    try:
        prior = json.load(open(legacy_cfg))
        if prior.get("smoke", False):
            print("Detected legacy smoke-test output in primary folder; clearing only that incomplete smoke run.")
            shutil.rmtree(OUT, ignore_errors=True)
    except Exception as exc:
        print("Could not inspect prior config; leaving outputs untouched:", exc)
run("train", "--mode", "dropout", "--epochs", EPOCHS, "--cache_rate", CACHE_RATE, "--resume", *COMMON)

$ /usr/bin/python3 modality_budget_experiment.py train --mode dropout --epochs 50 --cache_rate 0.3 --resume --data_root /kaggle/temp/brats --out_dir /kaggle/working/mb_exp --max_cases 400 --roi 128
cases=400 train=280 val=40 test=80 gpus=2

Loading dataset: 100%|██████████| 84/84 [00:49<00:00,  1.71it/s]
[dropout] epoch 1/50 loss=0.8305 val=nan best=-1.0000 (157s)
[dropout] epoch 2/50 loss=0.7872 val=nan best=-1.0000 (150s)
[dropout] epoch 3/50 loss=0.7473 val=nan best=-1.0000 (157s)
[dropout] epoch 4/50 loss=0.7147 val=nan best=-1.0000 (150s)
[dropout] epoch 5/50 loss=0.6603 val=0.5277 best=0.5277 (210s)
[dropout] epoch 6/50 loss=0.6064 val=nan best=0.5277 (155s)
[dropout] epoch 7/50 loss=0.5525 val=nan best=0.5277 (152s)
[dropout] epoch 8/50 loss=0.4995 val=nan best=0.5277 (156s)
[dropout] epoch 9/50 loss=0.4421 val=nan best=0.5277 (156s)
[dropout] epoch 10/50 loss=0.4159 val=0.6167 best=0.6167 (218s)
[dropout] epoch 11/50 loss=0.3800 val=nan best=0.6167 (151s)
[dropout] epoch 12/50 

## 8. Train or resume the full-input baseline (the control)
This shares the same case split as the dropout model. The `--resume` option preserves a completed run; if the model is already complete, you can skip this cell and proceed to evaluation.


In [ ]:
run("train", "--mode", "full", "--epochs", EPOCHS, "--cache_rate", CACHE_RATE, "--resume", *COMMON)

## 9. Evaluate all 15 sequence subsets on the held-out test cases
Run this after training. This pass computes Dice and stratified-sample ECE/Brier scores for all regions and subsets. Add `--hd95` to compute HD95 using NIfTI voxel spacing (known spatial units are converted to millimetres; unspecified units use voxels). Existing model checkpoints are reused; training is not repeated here.


In [ ]:
run("evaluate", "--mode", "dropout", "--hd95", *COMMON)
run("evaluate", "--mode", "full", "--hd95", *COMMON)

## 10. Analysis: Shapley, budget curves, minimal sets, safety metrics, paired inference
The relative sequence costs remain placeholders. The analysis now sweeps the T1ce cost from 1 to 10, reports the cost-free fewest-sequence solutions, summarizes HD95/ECE/Brier with case-level bootstrap intervals, and compares dropout versus full-input using paired case-level bootstrap CIs plus sign-flip permutation tests with Holm correction. These are analyses of your held-out predictions; they do not require retraining.


In [ ]:
COSTS = {"T1": 1, "T1ce": 3, "T2": 1, "FLAIR": 1}
json.dump(COSTS, open("costs.json", "w"))
res = [f"dropout={OUT}/results_dropout.csv", f"full={OUT}/results_full.csv"]
run("analyze", "--out_dir", OUT, "--n_boot", 2000, "--n_perm", 10000, "--costs", "costs.json", "--results", *res)
# Empty-ground-truth sensitivity analysis (optional, separate analysis folder).
run("analyze", "--out_dir", f"{OUT}/excl", "--n_boot", 2000, "--n_perm", 10000, "--empty_policy", "exclude",
    "--costs", "costs.json", "--results", *res)

## 11. View figures and tables

In [ ]:
from IPython.display import Image, display
import pandas as pd
A = f"{OUT}/analysis"
for f in ["fig_shapley_dropout.png", "fig_shapley_full.png", "fig_budget_curves.png", "fig_subset_heatmap_dropout.png"]:
    display(Image(f"{A}/{f}"))
for f in ["shapley_dropout.csv", "shapley_full.csv", "minimal_sets_dropout.csv", "paired_bootstrap_comparison.csv", "cost_sensitivity_transitions.csv", "cost_free_minimal_sets.csv", "safety_metrics_dropout.csv", "safety_metrics_full.csv"]:
    p = f"{A}/{f}"
    if os.path.exists(p):
        print("\n", f); display(pd.read_csv(p).head(30).round(4))
print(open(f"{A}/shapley_dropout.tex").read())

## 12. Optional three-seed validation (recommended for the conference version)
This is compute-intensive: it trains both regimes for seeds 42, 43, and 44. Keep `RUN_MULTI_SEED = False` until the single-seed rerun and safety analysis complete. The new `--split_seed 42` keeps the same 400-case selection and train/validation/test split while varying model-training randomness. Once all six runs finish, this cell reports across-seed variability and runs the same analysis on per-case metrics averaged across seeds. The averaged metrics are not an ensemble prediction.


In [ ]:
RUN_MULTI_SEED = False  # Set True to train/resume all seed runs. This may take many hours.
SEEDS = [42, 43, 44]
OUT_SEEDS = "/kaggle/working/mb_exp_multiseed"
os.makedirs(OUT_SEEDS, exist_ok=True)
if RUN_MULTI_SEED:
    for seed in SEEDS:
        seed_out = f"{OUT_SEEDS}/seed_{seed}"
        common_seed = ["--data_root", DATA, "--out_dir", seed_out, "--max_cases", MAX_CASES,
                       "--roi", ROI, "--seed", seed, "--split_seed", 42]
        run("train", "--mode", "dropout", "--epochs", EPOCHS, "--cache_rate", CACHE_RATE, "--resume", *common_seed)
        run("train", "--mode", "full", "--epochs", EPOCHS, "--cache_rate", CACHE_RATE, "--resume", *common_seed)
        run("evaluate", "--mode", "dropout", "--hd95", *common_seed)
        run("evaluate", "--mode", "full", "--hd95", *common_seed)

# Aggregate only after result CSVs for every seed and both modes exist.
seed_csvs = {(seed, mode): f"{OUT_SEEDS}/seed_{seed}/results_{mode}.csv"
             for seed in SEEDS for mode in ["dropout", "full"]}
missing = [p for p in seed_csvs.values() if not os.path.exists(p)]
if missing:
    print(f"Multi-seed aggregation skipped; {len(missing)} result files are missing. Complete the runs above first.")
else:
    all_seed_rows = []
    for (seed, mode), path in seed_csvs.items():
        d = pd.read_csv(path)
        d["seed"] = seed; d["model"] = mode
        all_seed_rows.append(d)
    seed_all = pd.concat(all_seed_rows, ignore_index=True)
    key_cols = ["case", "mask", "subset", "k"]
    metric_cols = [c for c in seed_all.columns if c.startswith(("dice_", "hd95_", "ece_", "brier_"))]
    seed_means = (seed_all.groupby(["seed", "model", "mask", "subset", "k"], as_index=False)[metric_cols]
                  .mean(numeric_only=True))
    seed_summary = (seed_means.groupby(["model", "mask", "subset", "k"])[metric_cols]
                    .agg(["mean", "std"]).reset_index())
    seed_summary.to_csv(f"{OUT_SEEDS}/multi_seed_summary.csv", index=False)
    print("Wrote", f"{OUT_SEEDS}/multi_seed_summary.csv")
    # Average per-case metrics over seeds, retaining one paired row per case/subset.
    for mode in ["dropout", "full"]:
        d = seed_all[seed_all["model"] == mode].copy()
        agg = {c: "mean" for c in metric_cols}
        for c in d.columns:
            if c not in key_cols + metric_cols + ["seed", "model"]:
                agg[c] = "first"
        avg = d.groupby(key_cols, as_index=False).agg(agg)
        avg.to_csv(f"{OUT_SEEDS}/results_{mode}_seedavg.csv", index=False)
    run("analyze", "--out_dir", f"{OUT_SEEDS}/seedavg", "--n_boot", 2000, "--n_perm", 10000,
        "--costs", "costs.json", "--results",
        f"dropout={OUT_SEEDS}/results_dropout_seedavg.csv",
        f"full={OUT_SEEDS}/results_full_seedavg.csv")


## 13. Package the results (download from the Output tab after *Save Version*)

In [ ]:
import zipfile
zip_path = f"{OUT}/results_package.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["analysis/*", "excl/analysis/*", "results_*.csv", "results_*_metadata.json", "split.json",
                "dropout/log.csv", "full/log.csv", "dropout/config.json", "full/config.json"]:
        for f in glob.glob(f"{OUT}/{pat}"):
            if os.path.isfile(f): z.write(f, os.path.relpath(f, OUT))
    seed_root = "/kaggle/working/mb_exp_multiseed"
    if os.path.isdir(seed_root):
        for pat in ["multi_seed_summary.csv", "results_*_seedavg.csv", "seed_*/*", "seed_*/dropout/config.json", "seed_*/dropout/log.csv", "seed_*/full/config.json", "seed_*/full/log.csv", "seedavg/analysis/*"]:
            for f in glob.glob(f"{seed_root}/{pat}"):
                if os.path.isfile(f): z.write(f, os.path.relpath(f, "/kaggle/working"))
print("created", zip_path, round(os.path.getsize(zip_path)/1e6, 1), "MB")


In [ ]:
import json, shutil
from pathlib import Path

# 1) Confirm the existing run really used seed 42 before reusing it
print("existing seed:", json.load(open(f"{OUT}/dropout/config.json")).get("seed"),
      json.load(open(f"{OUT}/full/config.json")).get("seed"))

# 2) Reuse the finished seed-42 run
OUT_SEEDS = "/kaggle/working/mb_exp_multiseed"
dst = Path(OUT_SEEDS) / "seed_42"; dst.mkdir(parents=True, exist_ok=True)
for f in ["results_dropout.csv", "results_full.csv", "split.json"]:
    if (Path(OUT) / f).exists() and not (dst / f).exists():
        shutil.copy(Path(OUT) / f, dst / f)

# 3) Train and evaluate the new seeds (same split_seed=42, so the split is identical)
for seed in [43, 44]:
    seed_out = f"{OUT_SEEDS}/seed_{seed}"
    cs = ["--data_root", DATA, "--out_dir", seed_out, "--max_cases", MAX_CASES,
          "--roi", ROI, "--seed", seed, "--split_seed", 42]
    run("train", "--mode", "dropout", "--epochs", EPOCHS, "--cache_rate", CACHE_RATE, "--resume", *cs)
    run("train", "--mode", "full",    "--epochs", EPOCHS, "--cache_rate", CACHE_RATE, "--resume", *cs)
    run("evaluate", "--mode", "dropout", "--hd95", *cs)
    run("evaluate", "--mode", "full",    "--hd95", *cs)

In [ ]:
import numpy as np, pandas as pd
from math import factorial as f
from pathlib import Path
SEEDS=[42,43,44]; OUT_SEEDS=Path("/kaggle/working/mb_exp_multiseed")
MODS=["T1","T1ce","T2","FLAIR"]; REG=["ET","TC","WT"]
COST={"T1":1,"T1ce":3,"T2":1,"FLAIR":1}; EPS=0.05

def sname(m): return "+".join(x for i,x in enumerate(MODS) if m>>i&1)
def load(seed,mode): return pd.read_csv(OUT_SEEDS/f"seed_{seed}"/f"results_{mode}.csv")
def vtab(df,r):
    v=df.groupby("mask")[f"dice_{r}"].mean().to_dict(); v[0]=0.0; return v
def shapley(v):
    phi=np.zeros(4)
    for m in range(4):
        for S in range(16):
            if S>>m&1: continue
            s=bin(S).count("1"); phi[m]+=f(s)*f(3-s)/f(4)*(v[S|1<<m]-v[S])
    return phi
def minimal_set(v):
    ok=[(sum(COST[MODS[i]] for i in range(4) if S>>i&1),-v[S],S)
        for S in range(1,16) if v[S]>=(1-EPS)*v[15]]
    return sname(min(ok)[2])

rows=[];sh=[];ms=[];hd=[]
for mode in ["dropout","full"]:
    for seed in SEEDS:
        df=load(seed,mode)
        for r in REG:
            v=vtab(df,r)
            for m in (15,13,2,8): rows.append(dict(model=mode,seed=seed,region=r,subset=sname(m),dice=v[m]))
            for i,p in enumerate(shapley(v)): sh.append(dict(model=mode,seed=seed,region=r,seq=MODS[i],phi=p))
            ms.append(dict(model=mode,seed=seed,region=r,min_set=minimal_set(v)))
            for m in (15,13,2):
                g=df[df["mask"]==m]; gt=g[f"gt_vox_{r}"]>0; pr=g[f"pred_vox_{r}"]>0
                h=pd.to_numeric(g[f"hd95_{r}"],errors="coerce"); ok=gt&pr&np.isfinite(h)
                hd.append(dict(model=mode,seed=seed,region=r,subset=sname(m),eligible=int(gt.sum()),
                    evaluable=int(ok.sum()),empty_pred=int((gt&~pr).sum()),
                    rate_pct=100*(gt&~pr).sum()/max(1,gt.sum()),hd95_mean=h[ok].mean() if ok.any() else np.nan))
D,S,M,H=map(pd.DataFrame,(rows,sh,ms,hd))
ms_sd=lambda x: f"{x.mean():.3f} ± {x.std(ddof=1):.3f}"
tabs={"dice":D.groupby(["model","region","subset"]).dice.agg(ms_sd).unstack("subset"),
      "shapley":S.groupby(["model","region","seq"]).phi.agg(ms_sd).unstack("seq"),
      "minimal_sets":M.pivot_table(index=["model","region"],columns="seed",values="min_set",aggfunc="first"),
      "hd95_audit":H.round(2)}
for k,t in tabs.items():
    print("\n##",k); print(t.to_string()); t.to_csv(OUT_SEEDS/f"seed_summary_{k}.csv")